## Required vs Optional Fields

Wann muss ein Wert angegeben werden, und wann darf er fehlen? In Pydantic entscheidet der Default-Wert: Ein Field ohne Default ist Pflicht (required), ein Field mit Default ist optional und darf beim Erzeugen weggelassen werden. Optional heißt dabei nur „darf fehlen“. Ob auch `None` erlaubt ist, ist eine eigene Frage für ein späteres Notebook.

**Danach ist klar:**
- woran Pydantic ein Pflicht-Field von einem optionalen Field unterscheidet,
- dass Default-Werte und spätere Zuweisungen standardmäßig nicht validiert werden,
- warum ein veränderlicher Default wie `[]` in einem Model unproblematisch ist.

Voraussetzung: Notebook `05 - Type Coercion`.

In [ ]:
# `BaseModel` definiert das Model, `ValidationError` fängt später den Fehlerfall ab.
from pydantic import BaseModel, ValidationError

In [ ]:
# Ohne Default-Werte sind `center` und `radius` Pflicht.
class Circle(BaseModel):
    center: tuple[int, int]
    radius: int

In [ ]:
# Beide Fields zeigen `required=True`: Pydantic erwartet beide Angaben.
Circle.model_fields

In [ ]:
# `center` bekommt den Default `(0, 0)`, `radius` bleibt Pflicht. Die neue Klasse ersetzt die alte unter demselben Namen.
class Circle(BaseModel):
    center: tuple[int, int] = (0,0)
    radius: int

In [ ]:
# `center` zeigt jetzt `required=False` samt Default, `radius` ist weiterhin Pflicht.
Circle.model_fields

**Vorhersage:** `center` hat jetzt einen Default. Lässt sich `Circle` mit `radius` allein erzeugen, und was steht dann in `center`?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Der Aufruf lässt `center` weg, und Pydantic setzt den Default `(0, 0)` ein. Fehlende Pflicht-Fields würden dagegen als `missing` gemeldet.

Kurz gesagt: Ein Default macht ein Field optional.
</details>

In [ ]:
Circle(radius=1)

In [ ]:
# Wird `center` ausdrücklich angegeben, entsteht dieselbe Instanz wie mit dem Default.
Circle(center=(0,0), radius=1)

In [ ]:
# Dictionary und JSON-Text ohne `center`: Fields mit Default dürfen im Input fehlen.
data = {"radius": 1}
data_json = '{"radius": 1}'

In [ ]:
# Auch `model_validate` setzt den Default für das fehlende `center` ein.
Circle.model_validate(data)

In [ ]:
# `model_validate_json` verhält sich bei JSON-Text genauso.
Circle.model_validate_json(data_json)

In [ ]:
# Ein übergebener Wert überschreibt den Default.
Circle(center=(1,1), radius=2)

In [ ]:
# Ein Default schützt nicht vor falschen Angaben: `"100"` ist kein Tuple, deshalb meldet Pydantic `tuple_type`.
try:
    Circle(center="100", radius=2)
except ValidationError as exc:
    print(exc)

In [ ]:
# Eine Instanz, bei der `center` vom Default stammt.
c = Circle(radius=2)
c

**Vorhersage:** Jetzt wird `"junk"` dem Field `center` zugewiesen. Löst das einen Fehler aus?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Keine Exception, `center` enthält danach `'junk'`. Wie schon in `02 - Creating a Pydantic Model` prüft Pydantic nur beim Erzeugen, nicht bei Zuweisungen.
</details>

In [ ]:
c.center = "junk"
c.center

Ein Default muss nicht zum Typ passen. Wo fällt das auf?

**Vorhersage:** `center` bekommt den Default `"junk"`, obwohl ein `tuple[int, int]` erwartet wird. Gibt es einen Fehler bei der Klassendefinition, bei `Circle(radius=2)` oder nirgends?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Nirgends: Pydantic validiert Defaults standardmäßig nicht, und `center` enthält `'junk'`. Mit `validate_default=True` würde das auffallen.

Kurz gesagt: Defaults werden wie Zuweisungen ungeprüft übernommen, solange es nicht eingeschaltet wird.
</details>

In [ ]:
class Circle(BaseModel):
    center: tuple[int, int] = "junk"
    radius: int

In [ ]:
Circle(radius=2)

### Veränderliche Defaults

In gewöhnlichen Python-Funktionen ist ein Default wie `[]` eine bekannte Falle, weil er nur einmal erzeugt wird. Wie verhält sich das in einem Pydantic-Model? Zuerst die Funktion zum Vergleich.

In [ ]:
# Der Default `[]` wird einmal beim Definieren der Funktion erzeugt, nicht bei jedem Aufruf neu.
# `int(time())` hängt die aktuelle Zeit in ganzen Sekunden an.
from time import time
def extend_list(user_list: list = []):
    user_list.append(int(time()))
    return user_list

In [ ]:
# Wird eine eigene Liste übergeben, bleibt der Default unberührt.
my_times = []
extend_list(my_times)
my_times

**Vorhersage:** `extend_list()` wird zweimal ohne Argument aufgerufen. Wie viele Einträge hat die Liste nach dem zweiten Aufruf, und hängen `my_times` und `my_new_times` zusammen?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Beide Aufrufe benutzen dieselbe Default-Liste, deshalb zeigt die Liste nach dem zweiten Aufruf zwei Einträge. `my_times` und `my_new_times` sind dasselbe Objekt, `my_times` hat also ebenfalls zwei Einträge.

Kurz gesagt: Ein veränderlicher Default in einer Funktion wird von allen Aufrufen geteilt.
</details>

In [ ]:
my_times = extend_list()
my_times

In [ ]:
my_new_times = extend_list()
my_new_times

**Vorhersage:** Im Model steht derselbe Default `[]`. Teilen sich zwei Instanzen die Liste, wenn an einer ein Element angehängt wird?

In [ ]:
class Model(BaseModel):
    my_list: list[int] = []


<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Pydantic kopiert veränderliche Defaults für jede Instanz. `m1` enthält das angehängte Element, `m2` bleibt leer.

Kurz gesagt: Im Model ist `[]` als Default sicher, anders als in einer Funktion.
</details>

In [ ]:
m1 = Model()
m2 = Model()
m1.my_list.append(1)
m1, m2

### Zusammenfassung

- Ein Field ohne Default ist Pflicht (`required=True`), ein Field mit Default darf fehlen. `model_fields` zeigt den Unterschied.
- Der Default gilt für Konstruktor, `model_validate` und `model_validate_json` gleichermaßen, ein übergebener Wert überschreibt ihn.
- Defaults und spätere Zuweisungen werden standardmäßig nicht validiert, Eingaben beim Erzeugen dagegen schon.
- Ein veränderlicher Default wie `[]` wird in einem Model pro Instanz kopiert und ist anders als in einer Funktion sicher.

Weiter mit `07 - Nullable Fields`.